In [1]:
import os
import json
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv

In [2]:
# .env 파일에서 환경변수 로드
load_dotenv()

True

In [14]:
# 허깅페이스 토큰 및 데이터셋 주소 수정포인트 본인 토큰 맞는지 확인 필요
hf_token = os.getenv('HF_TOKEN')
hf_repo = os.getenv('HF_DATASET_REPO', 'yunhwa/ai_question')

In [ ]:
from pathlib import Path

# 노트북 위치 기준으로 경로 설정
notebook_dir = Path(os.getcwd())
data_root = notebook_dir.parent / 'data' / 'raw'
processed_dir = notebook_dir.parent / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"data_root   : {data_root}")
print(f"processed_dir: {processed_dir}")

# data/raw 하위 JSON 파일 재귀 수집
json_files = sorted(data_root.rglob('*.json'))
print(f"JSON 파일 수: {len(json_files):,}")


In [ ]:
all_rows = []
error_files = []

for json_path in json_files:
    # raw/ 기준 최상위 폴더명의 첫 단어를 input 값으로 사용
    # 예: 형사법_라벨링데이터 → 형사법
    top_folder = json_path.relative_to(data_root).parts[0]
    input_val = top_folder.split('_')[0]

    try:
        with open(json_path, 'r', encoding='utf-8-sig') as f:
            data = json.load(f)
    except UnicodeDecodeError:
        try:
            with open(json_path, 'r', encoding='ms949') as f:
                data = json.load(f)
        except Exception as e:
            error_files.append((str(json_path), str(e)))
            continue
    except Exception as e:
        error_files.append((str(json_path), str(e)))
        continue

    # JSON 구조 분기
    # 형사법·행정법 → data['label']['input'] / ['output']
    # 민사법·지식재산권법 → data['taskinfo']['input'] / ['output']
    try:
        if 'label' in data:
            section = data['label']
        elif 'taskinfo' in data:
            section = data['taskinfo']
        else:
            error_files.append((str(json_path), f"알 수 없는 키: {list(data.keys())}"))
            continue

        instruction = section.get('input', '').strip()
        output = section.get('output', '').strip()

        if instruction and output:
            all_rows.append({
                'instruction': instruction,
                'input': input_val,
                'output': output,
            })
    except Exception as e:
        error_files.append((str(json_path), str(e)))

pdf = pd.DataFrame(all_rows)
print(f"총 샘플 수  : {len(pdf):,}")
print(f"오류 파일 수: {len(error_files):,}")
print(f"\n법률 도메인별 분포:\n{pdf['input'].value_counts().to_string()}")
pdf.head()


In [ ]:
# 알파카 포맷 JSON으로 저장
output_path = processed_dir / 'alpaca_legal.json'
pdf.to_json(output_path, orient='records', force_ascii=False, indent=2)

size_mb = output_path.stat().st_size / 1024 / 1024
print(f"저장 완료 : {output_path}")
print(f"파일 크기 : {size_mb:.1f} MB")


In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(pdf)

print(f"허깅페이스 허브({hf_repo})에 업로드 시도...")
hf_dataset.push_to_hub(hf_repo, token=hf_token)
print("허깅페이스 업로드 완료!")
